![SGSSS Logo](https://github.com/SGSSSonline/text-analysis/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 3: Topic Modelling

Topic models are one of the most widely used unsupervised methods in computational text analysis. They allow researchers to discover the latent thematic structure of a corpus — that is, to identify the topics that documents are about — without requiring any prior labelling or classification.

In this practical, we work through the full topic modelling pipeline: from preprocessing and representation, through model fitting and interpretation, to evaluation and visualisation. We use **Latent Dirichlet Allocation (LDA)**, the foundational topic model introduced by Blei, Ng, and Jordan (2003), which remains a workhorse of applied text analysis in the social sciences.

## Aims

This practical has two aims:

1. **Demonstrate how to use Python to fit, interpret, and evaluate topic models** on social science text data.
2. **Cultivate computational thinking skills** — understanding the choices involved in topic modelling and how they affect substantive conclusions.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~75 minutes |
| **Pre-requisites** | Practical 1 (Preprocessing Text) |
| **Learning outcomes** | Understand the intuition behind Latent Dirichlet Allocation (LDA) |
| | Be able to prepare text data for topic modelling using gensim |
| | Be able to fit and interpret an LDA model |
| | Be able to visualise topics using pyLDAvis and matplotlib |
| | Be able to examine document-topic assignments |
| | Understand how to evaluate topic models using coherence scores |
| | Appreciate the effect of preprocessing choices on topic model outputs |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain Python code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or modules defined in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
print("Hello! What is your name?")
name = input()
print(f"Welcome to the course, {name}!")

## Setup

Before we begin, we need to install and import the Python libraries we will use throughout this practical. These include:

- **NLTK** (Natural Language Toolkit) — for tokenisation and stopword removal.
- **gensim** — for building the dictionary, bag-of-words corpus, and fitting LDA topic models.
- **pyLDAvis** — for interactive visualisation of topic models.
- **matplotlib** — for static plots and charts.
- **pandas** — for working with tabular data.

In [ ]:
# Install packages (if needed on Colab)
!pip install nltk gensim pyLDAvis matplotlib pandas -q

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Successfully imported necessary modules")

## Loading the Corpus

We will work with the same sample of 20 UK parliamentary speeches from the **Hansard debates** that we used in Practical 1. These speech excerpts cover a range of policy topics debated in the House of Commons, including health, the economy, climate, immigration, education, housing, and defence.

> **A note on corpus size:** Our sample corpus is small (20 speeches). In practice, topic models work best with larger corpora (hundreds or thousands of documents). We use this small sample for learning purposes — the code scales to larger datasets. With a small corpus, topics may be less clearly separated and coherence scores less reliable. Keep this in mind as you interpret the results.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
speeches_data = {
    "date": [
        "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
        "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
        "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
        "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
    ],
    "speaker": [
        "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
        "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
        "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
        "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
        "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
    ],
    "party": [
        "Labour", "Conservative", "Labour", "Conservative", "Labour",
        "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
        "SNP", "Labour", "Conservative", "Labour", "SNP",
        "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
    ],
    "speech_text": [
        "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
        "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
        "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
        "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
        "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
        "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
        "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
        "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
        "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
        "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
        "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
        "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
        "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
        "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
        "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
        "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
        "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
        "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
        "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
        "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
    ]
}

# Create a pandas DataFrame from the dictionary
df = pd.DataFrame(speeches_data)

print(f"Dataset shape: {df.shape[0]} speeches, {df.shape[1]} columns")
df.head()

## Preprocessing

Before fitting a topic model, we need to preprocess the text. This is a brief recap of the steps covered in Practical 1. We will:

1. **Tokenise** each speech into individual words.
2. **Lowercase** all tokens.
3. **Remove punctuation** and non-alphabetic tokens.
4. **Remove stopwords** — common words that carry little substantive meaning.

We define a preprocessing function and apply it to every speech in the corpus. The output is a list of cleaned tokens for each document — this is the format that gensim requires for building a topic model.

In [ ]:
def preprocess(text):
    """Apply preprocessing steps to a single text string."""
    # Tokenise
    tokens = word_tokenize(text)
    # Lowercase and remove punctuation/numbers
    tokens = [t.lower() for t in tokens if t.isalpha()]
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

# Apply preprocessing to all speeches
tokenized_docs = df["speech_text"].apply(preprocess).tolist()

# View the first preprocessed document
print(f"Speaker: {df['speaker'].iloc[0]}")
print(f"Original: {df['speech_text'].iloc[0][:100]}...")
print(f"Preprocessed tokens: {tokenized_docs[0]}")
print(f"\nNumber of tokens: {len(tokenized_docs[0])}")

Notice how the preprocessing has stripped away grammatical scaffolding — articles, prepositions, auxiliary verbs — and left us with the substantive content words. These are the words that will drive the topic model.

## Preparing for LDA

Latent Dirichlet Allocation (LDA) does not work directly with raw text or even with lists of tokens. It requires two things:

1. A **dictionary** — a mapping from words to unique integer IDs.
2. A **bag-of-words (BoW) corpus** — each document represented as a list of (word_id, count) tuples.

The bag-of-words representation discards word order entirely. Each document becomes a vector of word frequencies. This is a deliberate simplification: topic models assume that the order of words does not matter for identifying themes — only which words appear and how often.

### Step 1: Create the Dictionary

The gensim `Dictionary` object scans all our preprocessed documents and assigns a unique integer ID to every distinct word in the corpus.

In [ ]:
# Create a gensim dictionary from the tokenized documents
dictionary = corpora.Dictionary(tokenized_docs)

print(f"Dictionary size (before filtering): {len(dictionary)} unique terms")
print(f"\nSample mappings (word -> ID):")
for word, word_id in list(dictionary.token2id.items())[:10]:
    print(f"  '{word}' -> {word_id}")

### Step 2: Filter Extremes

Not all words are equally useful for topic modelling. Words that appear in only one document are too rare to reveal shared themes. Words that appear in almost every document are too common to distinguish between topics. We filter both out:

- `no_below=2`: Remove words that appear in fewer than 2 documents.
- `no_above=0.8`: Remove words that appear in more than 80% of documents.

This is analogous to the TF-IDF logic from Practical 1: we want words that are frequent enough to matter but distinctive enough to differentiate.

In [ ]:
# Filter extremes
dictionary.filter_extremes(no_below=2, no_above=0.8)

print(f"Dictionary size (after filtering): {len(dictionary)} unique terms")
print(f"\nRemaining terms (sample):")
for word_id in list(dictionary.keys())[:15]:
    print(f"  ID {word_id}: '{dictionary[word_id]}'")

The filtering has reduced the vocabulary. This is expected and desirable: a smaller, more focused vocabulary helps the topic model identify clearer themes.

### Step 3: Create the Bag-of-Words Corpus

Now we convert each document from a list of tokens into a bag-of-words representation using the filtered dictionary. Each document becomes a list of `(word_id, count)` tuples.

In [ ]:
# Create bag-of-words corpus
bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

# View the BoW representation of the first document
print(f"Speaker: {df['speaker'].iloc[0]}")
print(f"\nBag-of-words representation (first document):")
print(bow_corpus[0])

print(f"\nHuman-readable version:")
for word_id, count in bow_corpus[0]:
    print(f"  '{dictionary[word_id]}': {count}")

Each tuple tells us: this word (identified by its ID) appears this many times in the document. This is all the information LDA needs. The order of words within a document is lost — hence "bag of words".

We now have everything we need to fit a topic model:
- A **dictionary** mapping words to IDs.
- A **bag-of-words corpus** where each document is a vector of word counts.

## Fitting Your First LDA Model

We are now ready to fit a **Latent Dirichlet Allocation (LDA)** model. LDA is a generative probabilistic model that assumes each document is a mixture of topics, and each topic is a distribution over words. The model learns both distributions simultaneously from the data.

The key parameters are:

| Parameter | Description |
| --- | --- |
| `corpus` | The bag-of-words corpus we created above |
| `id2word` | The dictionary mapping word IDs to words |
| `num_topics` | The number of topics to discover (K) — we start with 3 |
| `random_state` | A seed for reproducibility — ensures the same results each time |
| `passes` | Number of passes through the entire corpus during training |
| `alpha` | Controls the document-topic distribution — `'auto'` lets the model learn it |
| `eta` | Controls the topic-word distribution — `'auto'` lets the model learn it |

We choose **K=3** as a starting point. With only 20 documents, a small number of topics is appropriate — we can experiment with other values later.

Setting `alpha='auto'` and `eta='auto'` allows gensim to optimise these hyperparameters during training, which generally produces better results than fixed values.

In [ ]:
# Fit an LDA model with 3 topics
lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=15,
    alpha='auto',
    eta='auto'
)

print("LDA model fitted successfully.")
print(f"Number of topics: {lda_model.num_topics}")
print(f"Vocabulary size: {len(dictionary)}")
print(f"Number of documents: {len(bow_corpus)}")

## Interpreting Topics

The model has identified 3 topics. Each topic is a probability distribution over the vocabulary: some words have a high probability under a given topic (they are strongly associated with it) and others have a low probability.

Let's examine the top 10 words for each topic. The output format is:

> `weight * "word" + weight * "word" + ...`

The **weight** (a decimal number) indicates how strongly associated a word is with the topic. Higher weights mean the word is more characteristic of that topic.

In [ ]:
# Display the top 10 words for each topic
topics = lda_model.print_topics(num_words=10)

for topic_id, topic_string in topics:
    print(f"Topic {topic_id}:")
    print(f"  {topic_string}")
    print()

### Reading the Output

Each topic is characterised by its most probable words. When interpreting topics, look for **coherent themes** — groups of words that relate to a common subject or concept.

For example, if a topic's top words include "health", "service", "patient", "waiting", you might label it as a **Health/NHS** topic. If another includes "economy", "tax", "trade", "business", you might call it an **Economy** topic.

Topic labelling is a qualitative, interpretive step. There is no algorithm that names topics for you — the researcher must read the top words and assign a meaningful label based on domain knowledge.

Let's create a cleaner view of the topics as a table.

In [ ]:
# Create a DataFrame of top words per topic
n_top_words = 10
topic_words = {}

for topic_id in range(lda_model.num_topics):
    top_words = lda_model.show_topic(topic_id, topn=n_top_words)
    topic_words[f"Topic {topic_id}"] = [f"{word} ({weight:.3f})" for word, weight in top_words]

topic_df = pd.DataFrame(topic_words)
topic_df.index = [f"Word {i+1}" for i in range(n_top_words)]
topic_df

**Reflection:** Look at the top words for each topic. Can you give each topic a meaningful label? What themes do you see? Remember that with only 20 documents the topics may be less sharply defined than they would be with a larger corpus.

## Visualising Topics

Visualisation is essential for interpreting topic models. We will use two approaches:

1. **pyLDAvis** — an interactive visualisation that shows the global topic structure and the words associated with each topic.
2. **matplotlib bar charts** — a static visualisation of the top words per topic.

### Interactive Visualisation with pyLDAvis

pyLDAvis creates an interactive display with two panels:

- **Left panel**: A scatterplot showing topics as circles. The size of each circle reflects how prevalent the topic is in the corpus. Topics that are far apart are more distinct from each other; topics that overlap share similar words.
- **Right panel**: A bar chart showing the most relevant words for the selected topic. The blue bars show the overall frequency of each word in the corpus; the red bars show the frequency within the selected topic.

You can click on topics in the left panel to explore them. The **relevance slider** (lambda) controls how words are ranked: at lambda=1, words are ranked by their probability within the topic; at lower values, words that are more exclusive to the topic are ranked higher.

In [ ]:
# Prepare and display pyLDAvis visualisation
pyLDAvis.enable_notebook()

vis_data = pyLDAvis.gensim_models.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.display(vis_data)

**Tips for exploring the visualisation:**

- Click on each topic circle in the left panel. What words characterise each topic?
- Adjust the relevance slider (lambda) to the left (e.g., 0.3). This highlights words that are more *exclusive* to a topic, even if they are not the most frequent. This can help clarify the meaning of a topic.
- Look at whether topics overlap. Overlapping topics share vocabulary and may represent similar themes.

### Static Bar Charts with matplotlib

Interactive visualisations are useful for exploration, but static plots are better for reports and presentations. Let's create a bar chart showing the top words and their weights for each topic.

In [ ]:
# Create bar charts of top words per topic
n_top_words = 10
fig, axes = plt.subplots(1, lda_model.num_topics, figsize=(5 * lda_model.num_topics, 5), sharey=False)

for topic_id in range(lda_model.num_topics):
    top_words = lda_model.show_topic(topic_id, topn=n_top_words)
    words = [word for word, _ in top_words]
    weights = [weight for _, weight in top_words]
    
    ax = axes[topic_id]
    ax.barh(range(n_top_words), weights, color='steelblue', edgecolor='white')
    ax.set_yticks(range(n_top_words))
    ax.set_yticklabels(words)
    ax.invert_yaxis()  # Highest weight at the top
    ax.set_xlabel('Weight')
    ax.set_title(f'Topic {topic_id}')

plt.suptitle('Top Words per Topic', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

These bar charts provide a clear, side-by-side comparison of the topics. The length of each bar indicates how strongly the word is associated with the topic. Words at the top of each chart are the most characteristic.

Compare this view with the pyLDAvis visualisation. Both tell the same story, but in different formats — the interactive tool is better for exploration, while the static charts are better for communication.

## Document-Topic Assignments

A key output of LDA is the **document-topic distribution** — for each document, the model estimates the proportion of the document that belongs to each topic. This is what makes LDA a "mixed membership" model: a single document can be about multiple topics in different proportions.

Let's examine the topic distribution for each speech and identify the **dominant topic** — the topic with the highest proportion in each document.

In [ ]:
# Display the dominant topic for each document
print(f"{'Doc':>4}  {'Speaker':<25} {'Party':<18} {'Dominant Topic':>14}  {'Proportion':>10}")
print("-" * 80)

for i, doc in enumerate(bow_corpus):
    topic_dist = lda_model.get_document_topics(doc)
    dominant = max(topic_dist, key=lambda x: x[1])
    print(f"{i:>4}  {df['speaker'].iloc[i]:<25} {df['party'].iloc[i]:<18} Topic {dominant[0]:>8}  {dominant[1]:>10.2f}")

The **proportion** column tells us how confident the model is in the assignment. A value close to 1.0 means the document is almost entirely about one topic. A value closer to 0.33 (with K=3) means the document is a mixture of all three topics in roughly equal proportions.

Let's also look at the full topic distribution for a few documents to see the mixture more clearly.

In [ ]:
# Show full topic distributions for the first 5 documents
print("Full topic distributions for the first 5 documents:\n")

for i in range(5):
    topic_dist = lda_model.get_document_topics(bow_corpus[i], minimum_probability=0)
    dist_str = "  ".join([f"Topic {t}: {p:.3f}" for t, p in topic_dist])
    print(f"Doc {i} ({df['speaker'].iloc[i]}):")
    print(f"  {dist_str}")
    print()

This mixed-membership property is one of the strengths of LDA over hard clustering methods. It reflects the reality that parliamentary speeches often touch on multiple themes — a speech about education might also mention the economy or public spending.

### Visualising Document-Topic Distributions

A heatmap can provide a useful overview of how topics are distributed across all documents in the corpus.

In [ ]:
# Build a document-topic matrix
doc_topic_matrix = []
for doc in bow_corpus:
    topic_dist = lda_model.get_document_topics(doc, minimum_probability=0)
    doc_topic_matrix.append([prob for _, prob in sorted(topic_dist)])

doc_topic_df = pd.DataFrame(
    doc_topic_matrix,
    columns=[f"Topic {i}" for i in range(lda_model.num_topics)],
    index=[f"{df['speaker'].iloc[i]} ({df['party'].iloc[i]})" for i in range(len(df))]
)

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(doc_topic_df.values, cmap='Blues', aspect='auto')

ax.set_xticks(range(lda_model.num_topics))
ax.set_xticklabels(doc_topic_df.columns)
ax.set_yticks(range(len(doc_topic_df)))
ax.set_yticklabels(doc_topic_df.index, fontsize=8)
ax.set_xlabel('Topic')
ax.set_title('Document-Topic Distributions')

plt.colorbar(im, ax=ax, label='Topic Proportion')
plt.tight_layout()
plt.show()

In the heatmap, darker cells indicate a higher proportion of the document assigned to that topic. Look for patterns: do speakers from the same party tend to cluster on the same topics? Are some topics more dominant across the corpus than others?

## Experimenting with K

So far we have fitted a model with K=3 topics. But how do we know 3 is the right number? In practice, **there is no single correct value of K**. The choice involves both statistical guidance and substantive judgement.

One commonly used statistical measure is **coherence**. Topic coherence measures how semantically similar the top words within each topic are. Higher coherence generally indicates more interpretable topics. The `c_v` coherence measure (Roder et al., 2015) is widely used and combines multiple coherence metrics.

Let's fit models with K=2, 3, 4, and 5, and compute the coherence score for each.

In [ ]:
# Fit models with different values of K and compute coherence
coherence_scores = []
k_values = [2, 3, 4, 5]

print("Fitting models and computing coherence scores...\n")

for k in k_values:
    model = LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=15
    )
    cm = CoherenceModel(
        model=model,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"  K={k}: Coherence = {score:.4f}")

print("\nDone.")

In [ ]:
# Plot coherence scores against K
plt.figure(figsize=(8, 5))
plt.plot(k_values, coherence_scores, marker='o', linewidth=2, markersize=8, color='steelblue')
plt.xlabel('Number of Topics (K)', fontsize=12)
plt.ylabel('Coherence Score (c_v)', fontsize=12)
plt.title('Topic Coherence by Number of Topics', fontsize=14)
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpreting the Coherence Plot

The coherence plot helps guide the choice of K, but it should not be treated as a definitive answer. Some things to keep in mind:

- **Higher coherence is generally better**, but the relationship is not always monotonic. Coherence can fluctuate, especially with small corpora.
- **Look for an "elbow"** — a point where coherence increases sharply and then levels off. This often suggests a good number of topics.
- **Substantive judgement matters**. Even if coherence is highest at K=2, it may be more informative to use K=3 or K=4 if the additional topics capture meaningful themes.
- **With a small corpus** (like our 20 speeches), coherence scores are less reliable. Do not over-interpret small differences.

As Grimmer, Roberts, and Stewart (2022) emphasise: "There is no right answer to the number of topics... the choice should be guided by the research question, the data, and the analyst's interpretation of the topics."

## The Effect of Preprocessing

Preprocessing choices — which words to keep, whether to stem or lemmatise, how aggressively to filter — **materially affect topic model outputs**. Two researchers analysing the same corpus with different preprocessing pipelines may arrive at different topics.

To illustrate this, let's fit a topic model on the same corpus but **without stopword removal** and compare the resulting topics to our original model.

In [ ]:
# Preprocess WITHOUT stopword removal
def preprocess_no_stopwords(text):
    """Preprocess text without removing stopwords."""
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens if t.isalpha()]
    return tokens

tokenized_docs_no_stop = df["speech_text"].apply(preprocess_no_stopwords).tolist()

# Build dictionary and corpus without stopword removal
dictionary_no_stop = corpora.Dictionary(tokenized_docs_no_stop)
dictionary_no_stop.filter_extremes(no_below=2, no_above=0.8)
bow_corpus_no_stop = [dictionary_no_stop.doc2bow(doc) for doc in tokenized_docs_no_stop]

print(f"Vocabulary with stopword removal:    {len(dictionary)} terms")
print(f"Vocabulary without stopword removal:  {len(dictionary_no_stop)} terms")

In [ ]:
# Fit LDA without stopword removal
lda_no_stop = LdaModel(
    corpus=bow_corpus_no_stop,
    id2word=dictionary_no_stop,
    num_topics=3,
    random_state=42,
    passes=15,
    alpha='auto',
    eta='auto'
)

# Compare topics side by side
print("=" * 70)
print("TOPICS WITH STOPWORD REMOVAL (our original model):")
print("=" * 70)
for topic_id, topic_string in lda_model.print_topics(num_words=8):
    print(f"  Topic {topic_id}: {topic_string}")

print()
print("=" * 70)
print("TOPICS WITHOUT STOPWORD REMOVAL:")
print("=" * 70)
for topic_id, topic_string in lda_no_stop.print_topics(num_words=8):
    print(f"  Topic {topic_id}: {topic_string}")

### What Do You Notice?

Compare the two sets of topics. Without stopword removal, common words like "the", "and", "of" dominate the topics, making them harder to interpret. The substantive content words are pushed down the list, and the topics become less distinct.

This demonstrates an important principle: **preprocessing choices materially affect topic model outputs**. The decisions you make about tokenisation, stopword removal, stemming, and filtering are not merely technical — they shape the substantive findings of your analysis.

In practice, it is good research practice to:
1. Report your preprocessing choices clearly.
2. Consider running the model with different preprocessing settings as a **robustness check**.
3. Be transparent about the fact that different choices may yield different results.

## Exercise

Now it is your turn. Your task is to:

1. **Fit a topic model with a different number of topics (K)**. Choose a value of K that we have not yet examined in detail (for example, K=4 or K=5).
2. **Compare the results to our K=3 model:**
   - (a) Are the topics more or less coherent? Use the coherence score to support your answer.
   - (b) Do you see any new themes emerge that were not visible with K=3?
   - (c) Which K do you prefer and why?

Use the skeleton code below to guide you. Replace the `# YOUR CODE HERE` comments with your own code.

In [ ]:
# Step 1: Choose your K and fit the model
my_k = ___  # Replace ___ with your chosen number of topics

# YOUR CODE HERE: fit an LDA model with your chosen K
# Hint: use the same bow_corpus and dictionary as before
my_model = LdaModel(
    # YOUR CODE HERE
)

In [ ]:
# Step 2: View the topics
# YOUR CODE HERE: print the topics from your model


In [ ]:
# Step 3: Compute the coherence score
# YOUR CODE HERE: create a CoherenceModel and get the coherence score


**Reflection:** Compare your model to the K=3 model. Answer the three questions above.

*YOUR ANSWER HERE*

## Appendix: Exercise Solution

In [ ]:
# --- Exercise Solution ---

# Step 1: Choose K and fit the model
my_k = 4

my_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=my_k,
    random_state=42,
    passes=15,
    alpha='auto',
    eta='auto'
)

print(f"Fitted LDA model with K={my_k} topics.")

In [ ]:
# Step 2: View the topics
print(f"Topics for K={my_k}:\n")
for topic_id, topic_string in my_model.print_topics(num_words=10):
    print(f"Topic {topic_id}: {topic_string}")
    print()

In [ ]:
# Step 3: Compute the coherence score
my_cm = CoherenceModel(
    model=my_model,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
my_coherence = my_cm.get_coherence()

# Also compute coherence for the original K=3 model for comparison
original_cm = CoherenceModel(
    model=lda_model,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
original_coherence = original_cm.get_coherence()

print(f"Coherence score for K=3: {original_coherence:.4f}")
print(f"Coherence score for K={my_k}: {my_coherence:.4f}")

if my_coherence > original_coherence:
    print(f"\nThe K={my_k} model has higher coherence.")
else:
    print(f"\nThe K=3 model has higher coherence.")

**Reflection:** With K=4, the model may split one of the K=3 topics into two more specific sub-themes, or it may produce a new topic that captures a theme not visible at K=3. Whether this is an improvement depends on whether the new topics are substantively meaningful and coherent. With our small corpus of 20 speeches, increasing K beyond 3 or 4 risks producing topics that are difficult to interpret — there simply is not enough data to support many distinct themes. In practice, the best K is the one that produces topics you can interpret, label, and use to answer your research question.

## Bibliography

- Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*, 3, 993-1022.
- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Roberts, M. E., Stewart, B. M., & Tingley, D. (2019). stm: An R Package for Structural Topic Models. *Journal of Statistical Software*, 91(2), 1-40.
- Roberts, M. E., Stewart, B. M., Tingley, D., Lucas, C., Leder-Luis, J., Gadarian, S. K., Albertson, B., & Rand, D. G. (2014). Structural Topic Models for Open-Ended Survey Responses. *American Journal of Political Science*, 58(4), 1064-1082.
- Roder, M., Both, A., & Hinneburg, A. (2015). Exploring the Space of Topic Coherence Measures. *Proceedings of the Eighth ACM International Conference on Web Search and Data Mining*, 399-408.

---

**END OF FILE**